# InsightVLM — NB7: DenseNet-121 + extra real + renderer aug

NB5x recipe (clean v2 renders, extra real pool, ~50% real batches, checkpoint on val real accuracy) plus renderer augmentation: each training chart is shown as its matplotlib, Plotly or randomised-style render at random. Vega-Lite stays unseen. One model: `d121_384_real50_xp`.

In [ ]:
import os, sys, json, time, random, warnings, gc
from pathlib import Path
from collections import Counter, defaultdict
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
import torchvision.models as M

SEED = 20260815
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device", DEVICE,
      "|", torch.cuda.device_count(), "GPU(s)")

DATA_ROOT = Path("/kaggle/input/datasets/asifxzaman/xxxyyy/dataset/dataset")
def find_data():
    if (DATA_ROOT / "records.jsonl").exists(): return DATA_ROOT
    # the dataset root is the records.jsonl next to tables/metadata.parquet: other attached
    # datasets carry their own records.jsonl, and Kaggle's mount paths change between runs
    c = [p.parent for p in Path("/kaggle/input").glob("**/records.jsonl")
         if (p.parent / "tables" / "metadata.parquet").exists()]
    if not c: raise FileNotFoundError("dataset root (records.jsonl + tables/) not found under /kaggle/input")
    return c[0]
DATA = find_data(); print("dataset:", DATA)

meta = pd.read_parquet(DATA / "tables" / "metadata.parquet")
_ci = json.loads((DATA / "class_index.json").read_text())["label_to_id"]
CLASSES = sorted(_ci, key=_ci.get); NC = len(CLASSES); C2I = {c: i for i, c in enumerate(CLASSES)}
tr = meta[meta.split == "train"].reset_index(drop=True)
va = meta[meta.split == "val"].reset_index(drop=True)
te = meta[meta.split == "test"].reset_index(drop=True)
print(f"{len(meta):,} samples | {NC} classes | train {len(tr):,} val {len(va):,} test {len(te):,}")

# ---- data pipeline -------------------------------------------------------
# No memmap cache. It was the source of two separate failures (a 24.6 GB file
# that was disk-bound, then a 21 GB one that was OOM-killed) and reading PNGs
# directly costs ~90 img/s, which is fine when the GPU is the bottleneck anyway.
NORM = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

def tf_train(size):
    # Label-preserving only. NO horizontal/vertical flip: mirroring turns
    # rel_strong_positive into rel_strong_negative and ts_increasing into
    # ts_decreasing, which are separate classes here.
    return T.Compose([
        T.Resize((size, size)),
        T.RandomApply([T.ColorJitter(0.25, 0.25, 0.2, 0.03)], p=0.7),
        T.RandomApply([T.RandomResizedCrop(size, scale=(0.85, 1.0), ratio=(0.95, 1.05))], p=0.5),
        T.RandomApply([T.GaussianBlur(3, (0.1, 1.2))], p=0.2),
        T.ToTensor(), NORM, T.RandomErasing(p=0.15, scale=(0.01, 0.04))])

def tf_eval(size):
    return T.Compose([T.Resize((size, size)), T.ToTensor(), NORM])

class ChartDS(Dataset):
    def __init__(self, df, size, train=False, crop_plot=False):
        self.df = df.reset_index(drop=True)
        self.tf = tf_train(size) if train else tf_eval(size)
        self.crop = crop_plot
        self.boxes = {}
        if crop_plot:
            for sp in ("train", "val", "test"):
                with open(DATA / "splits" / f"{sp}.jsonl") as fh:
                    for line in fh:
                        r = json.loads(line)
                        b = next((x["bbox"] for x in r["render"]["grounding_boxes"]
                                  if x["label"] == "plot_area"), None)
                        if b: self.boxes[r["sample_id"]] = b
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(DATA / r.image).convert("RGB")
        if self.crop:
            b = self.boxes.get(r.sample_id)
            if b:
                x0, y0, x1, y1 = [int(round(v)) for v in b]
                if x1 - x0 > 20 and y1 - y0 > 20: img = img.crop((x0, y0, x1, y1))
        return self.tf(img), C2I[r.label], i

def loader(df, size, train, bs=48, crop=False):
    # modest workers, no pinned memory, no persistent workers: the combination
    # of persistent_workers + pin_memory leaked ~5 GB per model
    return DataLoader(ChartDS(df, size, train, crop), batch_size=bs, shuffle=train,
                      num_workers=2, pin_memory=False, prefetch_factor=2,
                      persistent_workers=False)

def build(name, n=NC):
    if name == "resnet50":
        m = M.resnet50(weights=M.ResNet50_Weights.IMAGENET1K_V2); m.fc = nn.Linear(m.fc.in_features, n)
    elif name == "resnet101":
        m = M.resnet101(weights=M.ResNet101_Weights.IMAGENET1K_V2); m.fc = nn.Linear(m.fc.in_features, n)
    elif name == "densenet121":
        m = M.densenet121(weights=M.DenseNet121_Weights.IMAGENET1K_V1)
        m.classifier = nn.Linear(m.classifier.in_features, n)
    elif name == "resnet18":
        m = M.resnet18(weights=M.ResNet18_Weights.IMAGENET1K_V1); m.fc = nn.Linear(m.fc.in_features, n)
    elif name == "convnext_tiny":
        m = M.convnext_tiny(weights=M.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, n)
    elif name == "efficientnet_v2_s":
        m = M.efficientnet_v2_s(weights=M.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n)
    else: raise ValueError(name)
    return m

def macro_f1(P, Y, n=NC):
    fs = []
    for i in range(n):
        tp = ((P==i)&(Y==i)).sum(); fp = ((P==i)&(Y!=i)).sum(); fn = ((P!=i)&(Y==i)).sum()
        pr = tp/(tp+fp) if tp+fp else 0; rc = tp/(tp+fn) if tp+fn else 0
        fs.append(2*pr*rc/(pr+rc) if pr+rc else 0)
    return float(np.mean(fs))

@torch.no_grad()
def predict(model, dl):
    model.eval(); P=[]; Y=[]; IX=[]
    for x, y, ix in dl:
        with torch.autocast("cuda", enabled=DEVICE.type=="cuda"):
            o = model(x.to(DEVICE, non_blocking=True))
        P.append(o.float().argmax(1).cpu()); Y.append(y); IX.append(ix)
    return torch.cat(P).numpy(), torch.cat(Y).numpy(), torch.cat(IX).numpy()

def train_model(name, size, epochs, bs=48, crop=False, patience=4, tag=""):
    dl_tr, dl_va = loader(tr, size, True, bs, crop), loader(va, size, False, bs, crop)
    model = build(name).to(DEVICE)
    if DEVICE.type == "cuda" and torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-4, epochs=epochs,
                                                steps_per_epoch=len(dl_tr), pct_start=0.25)
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type=="cuda")
    ckpt = f"{name}{tag}_best.pt"; best = -1.0; stale = 0; hist = []
    for ep in range(1, epochs+1):
        model.train(); t0 = time.time(); tot=n=corr=0
        for bi, (x, y, _) in enumerate(dl_tr):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=DEVICE.type=="cuda"):
                out = model(x); loss = crit(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.detach().item()*y.size(0); corr += (out.detach().argmax(1)==y).sum().item(); n += y.size(0)
            if bi % 60 == 0:
                print(f"\r  ep{ep} [{bi}/{len(dl_tr)}] loss {tot/max(n,1):.3f} acc {corr/max(n,1):.3f}", end="", flush=True)
        P, Y, _ = predict(model, dl_va); vf1 = macro_f1(P, Y)
        hist.append(dict(epoch=ep, loss=tot/n, train_acc=corr/n, val_acc=float((P==Y).mean()), val_f1=vf1))
        print(f"\r  ep{ep}  loss {tot/n:.3f}  train {corr/n:.3f}  val_acc {(P==Y).mean():.4f}  val_F1 {vf1:.4f}  ({time.time()-t0:.0f}s)")
        if vf1 > best:
            best, stale = vf1, 0
            torch.save((model.module if isinstance(model, nn.DataParallel) else model).state_dict(), ckpt)
        else:
            stale += 1
            if stale >= patience:
                print(f"  early stop: val macro-F1 flat for {patience} epochs"); break
    clean = build(name).to(DEVICE); clean.load_state_dict(torch.load(ckpt)); clean.eval()
    del model, dl_tr, dl_va, opt, sched, scaler; gc.collect(); torch.cuda.empty_cache()
    return clean, hist, best, ckpt

def save(obj, path):
    Path(path).write_text(json.dumps(obj, indent=2, default=str))
    print("wrote", path)
print("preamble ready")

In [ ]:
RENDER_SRC = '"""\n06_render.py — Render each record to a chart image with pixel-valid grounding boxes.\n\nTwo rendering rules that the notebook prototype got wrong, and which matter:\n\n  1. NO bbox_inches="tight". Tight-cropping recomputes the canvas *after* the\n     axes transform has been read, so every box drawn from ax.transData is\n     shifted relative to the file that actually gets saved. The canvas here is\n     fixed, so display coordinates and image pixels are the same thing.\n  2. Boxes are emitted in image pixel space with the y axis flipped, because\n     matplotlib\'s origin is bottom-left and an image\'s is top-left.\n\nEach image carries evidence boxes marking the pixels that support the finding:\nthe outlier band, the fitted trend, the changepoint, the anomalous points, the\ndominant wedge. That grounding is what separates this from chart-QA datasets.\n"""\nimport argparse\nimport json\nimport os\nimport sys\nimport warnings\nimport zlib\nfrom collections import Counter\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt          # noqa: E402\nimport numpy as np                       # noqa: E402\nfrom matplotlib.gridspec import GridSpec  # noqa: E402\nfrom tqdm import tqdm                    # noqa: E402\n\nwarnings.filterwarnings("ignore")\n\nROOT = Path(__file__).resolve().parent.parent\nCACHE = ROOT / "cache"\nDATA = ROOT / "dataset"\n# env, not globals set in main(): spawned worker processes re-import this module\nIMG = Path(os.environ.get("RENDER_OUT", DATA / "images"))\n\nW_PX, H_PX, DPI = 768, 576, 100\nFIGSIZE = (W_PX / DPI, H_PX / DPI)\n\nSTYLES = [\n    ("seaborn-v0_8-whitegrid", "tab10"), ("seaborn-v0_8-darkgrid", "Set2"),\n    ("ggplot", "tab10"), ("bmh", "Dark2"), ("fivethirtyeight", "tab10"),\n    ("seaborn-v0_8-ticks", "colorblind"), ("default", "tab10"),\n    ("seaborn-v0_8-muted", "Set1"),\n]\n\n\n# --clean: draw no label-dependent marks. The trend line (only on increasing/\n# decreasing), changepoint line and anomaly circles each reveal the class, so a\n# model can read the annotation instead of the data. Boxes are still emitted.\nCLEAN = os.environ.get("RENDER_CLEAN") == "1"\n# --randstyle: training-only domain randomisation (fonts, line/marker sizes, colours,\n# grids, spines, backgrounds, titles). The data and the chart type are untouched, so\n# the label is too. Its own seed, so the same chart gets a different look.\nRANDSTYLE = os.environ.get("RENDER_RANDSTYLE") == "1"\nRAND_CMAPS = ["tab10", "Set1", "Set2", "Dark2", "Paired", "viridis", "plasma", "cividis", "Accent"]\n\n\ndef _rand_rc(rr):\n    import colorsys\n    tint = lambda: colorsys.hsv_to_rgb(rr.uniform(0, 1), rr.uniform(0, 0.12), rr.uniform(0.93, 1.0))  # noqa: E731\n    return {\n        "font.family": str(rr.choice(["DejaVu Sans", "DejaVu Serif", "DejaVu Sans Mono", "STIXGeneral"])),\n        "font.size": float(rr.uniform(7, 13)), "lines.linewidth": float(rr.uniform(0.7, 3.2)),\n        "lines.markersize": float(rr.uniform(2, 8)), "axes.linewidth": float(rr.uniform(0.4, 2.0)),\n        "axes.grid": bool(rr.random() < 0.5), "grid.alpha": float(rr.uniform(0.2, 0.9)),\n        "grid.linestyle": str(rr.choice(["-", "--", ":"])), "axes.facecolor": tint(),\n        "figure.facecolor": tint() if rr.random() < 0.3 else "white",\n        "axes.spines.top": bool(rr.random() < 0.5), "axes.spines.right": bool(rr.random() < 0.5),\n        "xtick.direction": str(rr.choice(["in", "out"])), "ytick.direction": str(rr.choice(["in", "out"])),\n    }\n\n\ndef _seed(sid):\n    # crc32, not hash(): str hashing is salted per process, so styles were not reproducible\n    return zlib.crc32(sid.encode()) % (2 ** 31)\n\n\ndef _px(fig, ax, xd, yd):\n    """Data coords -> image pixel coords (origin top-left)."""\n    pts = ax.transData.transform(np.column_stack([np.atleast_1d(xd), np.atleast_1d(yd)]))\n    return pts[:, 0], H_PX - pts[:, 1]\n\n\ndef _box_from_data(fig, ax, x0, y0, x1, y1, label):\n    xs, ys = _px(fig, ax, [x0, x1], [y0, y1])\n    a, b = sorted([float(xs[0]), float(xs[1])])\n    c, d = sorted([float(ys[0]), float(ys[1])])\n    return {"label": label,\n            "bbox": [round(max(0, a), 1), round(max(0, c), 1),\n                     round(min(W_PX, b), 1), round(min(H_PX, d), 1)]}\n\n\ndef _box_from_artist(fig, artist, label):\n    try:\n        e = artist.get_window_extent(fig.canvas.get_renderer())\n    except Exception:\n        return None\n    return {"label": label,\n            "bbox": [round(max(0, float(e.x0)), 1), round(max(0, H_PX - float(e.y1)), 1),\n                     round(min(W_PX, float(e.x1)), 1), round(min(H_PX, H_PX - float(e.y0)), 1)]}\n\n\n# --------------------------------------------------------------------------\n# per-arity drawing. Each returns a list of (kind, args) evidence requests\n# resolved to pixel boxes after fig.canvas.draw().\n# --------------------------------------------------------------------------\ndef draw_univariate(ax, rec, r, colour):\n    st = rec["ground_truth"]["statistics"]\n    x = np.asarray(rec["ground_truth"]["data_table"]["x"], float)\n    ct = rec["chart_type"]\n    ev = []\n\n    if ct == "histogram":\n        nb = int(np.clip(np.sqrt(len(x)) * r.uniform(0.8, 1.4), 12, 45))\n        cnt, edges, _ = ax.hist(x, bins=nb, color=colour, edgecolor="white", alpha=0.88)\n        i = int(np.argmax(cnt))\n        ev.append(("data", edges[i], 0, edges[i + 1], cnt[i], "modal_bin"))\n        ax.set_ylabel("Frequency")\n    elif ct == "kde":\n        from scipy.stats import gaussian_kde\n        gr = np.linspace(x.min(), x.max(), 240)\n        dens = gaussian_kde(x)(gr)\n        ax.fill_between(gr, dens, color=colour, alpha=0.45)\n        ax.plot(gr, dens, color=colour, lw=2)\n        ev.append(("data", gr[int(np.argmax(dens))], 0, gr[int(np.argmax(dens))], dens.max(), "peak"))\n        ax.set_ylabel("Density")\n    elif ct == "boxplot":\n        bp = ax.boxplot(x, vert=True, patch_artist=True, widths=0.5)\n        bp["boxes"][0].set_facecolor(colour)\n        bp["boxes"][0].set_alpha(0.8)\n        ev.append(("data", 0.75, st["q1"], 1.25, st["q3"], "iqr_box"))\n        ev.append(("data", 0.6, st["median"], 1.4, st["median"], "median_line"))\n        ax.set_ylabel(rec["axis_labels"]["x"])\n        ax.set_xticks([])\n    elif ct == "violin":\n        vp = ax.violinplot(x, showmedians=True, widths=0.7)\n        for b in vp["bodies"]:\n            b.set_facecolor(colour)\n            b.set_alpha(0.7)\n        ev.append(("data", 0.7, st["q1"], 1.3, st["q3"], "iqr_region"))\n        ax.set_ylabel(rec["axis_labels"]["x"])\n        ax.set_xticks([])\n    elif ct == "ecdf":\n        s = np.sort(x)\n        ax.step(s, np.arange(1, len(s) + 1) / len(s), color=colour, lw=2)\n        ev.append(("data", st["median"], 0, st["median"], 0.5, "median_crossing"))\n        ax.set_ylabel("Cumulative proportion")\n    elif ct == "qq":\n        from scipy import stats as sps\n        (osm, osr), (sl, ic, _) = sps.probplot(x, dist="norm")\n        ax.scatter(osm, osr, s=14, color=colour, alpha=0.75)\n        ax.plot(osm, sl * osm + ic, "r--", lw=1.8)\n        ev.append(("data", osm.min(), (sl * osm + ic).min(), osm.max(), (sl * osm + ic).max(),\n                   "normal_reference_line"))\n        ax.set_xlabel("Theoretical quantiles")\n        ax.set_ylabel("Sample quantiles")\n    elif ct == "boxen":\n        import seaborn as sns\n        sns.boxenplot(y=x, color=colour, ax=ax)\n        ev.append(("data", -0.4, st["q1"], 0.4, st["q3"], "iqr_box"))\n        ax.set_ylabel(rec["axis_labels"]["x"])\n    else:  # strip\n        jitter = r.normal(0, 0.045, len(x))\n        ax.scatter(jitter, x, s=15, color=colour, alpha=0.55)\n        ax.set_xlim(-0.35, 0.35)\n        ev.append(("data", -0.3, st["q1"], 0.3, st["q3"], "iqr_region"))\n        ax.set_ylabel(rec["axis_labels"]["x"])\n        ax.set_xticks([])\n\n    if ct in ("histogram", "kde", "ecdf"):\n        ax.set_xlabel(rec["axis_labels"]["x"])\n    lo, hi = st.get("iqr_lower_fence"), st.get("iqr_upper_fence")\n    if lo is not None and st.get("outlier_count", 0) > 0:\n        if ct in ("histogram", "kde", "ecdf"):\n            if x.max() > hi:\n                ev.append(("data", hi, ax.get_ylim()[0], x.max(), ax.get_ylim()[1], "upper_outliers"))\n            if x.min() < lo:\n                ev.append(("data", x.min(), ax.get_ylim()[0], lo, ax.get_ylim()[1], "lower_outliers"))\n    return ev\n\n\ndef draw_bivariate(ax, rec, r, colour):\n    st = rec["ground_truth"]["statistics"]\n    d = rec["ground_truth"]["data_table"]\n    x, y = np.asarray(d["x"], float), np.asarray(d["y"], float)\n    ct = rec["chart_type"]\n    ev = []\n\n    if ct == "hexbin":\n        ax.hexbin(x, y, gridsize=int(r.integers(16, 28)), cmap="viridis", mincnt=1)\n    elif ct == "bubble":\n        sz = np.abs(r.normal(45, 18, len(x))).clip(8, 130)\n        ax.scatter(x, y, s=sz, color=colour, alpha=0.5, edgecolor="white", lw=0.5)\n    else:\n        ax.scatter(x, y, s=int(r.integers(14, 30)), color=colour, alpha=0.65,\n                   edgecolor="white", lw=0.4)\n\n    if ct == "regplot" or (ct == "scatter" and r.random() < 0.45):\n        b1, b0 = st.get("linear_slope"), st.get("linear_intercept")\n        if b1 is not None:\n            gx = np.linspace(x.min(), x.max(), 60)\n            ax.plot(gx, b1 * gx + b0, "r-", lw=2, alpha=0.9)\n            ev.append(("data", gx[0], b1 * gx[0] + b0, gx[-1], b1 * gx[-1] + b0, "fitted_trend"))\n\n    cls = rec["label"]\n    if cls == "rel_heteroscedastic":\n        q = np.quantile(x, [2 / 3, 1.0])\n        ev.append(("data", q[0], y.min(), q[1], y.max(), "high_variance_region"))\n        q2 = np.quantile(x, [0.0, 1 / 3])\n        ev.append(("data", q2[0], y.min(), q2[1], y.max(), "low_variance_region"))\n    elif cls == "rel_nonlinear_nonmonotonic":\n        c2 = st.get("curvature_coef") or 0\n        xs = (x - x.mean()) / (x.std() + 1e-12)\n        vi = int(np.argmin(xs ** 2))\n        ev.append(("data", x[vi] - 0.1 * x.std(), y.min(), x[vi] + 0.1 * x.std(), y.max(),\n                   "curvature_vertex" if c2 else "vertex"))\n\n    ax.set_xlabel(rec["axis_labels"]["x"])\n    ax.set_ylabel(rec["axis_labels"]["y"])\n    return ev\n\n\ndef draw_timeseries(ax, rec, r, colour):\n    st = rec["ground_truth"]["statistics"]\n    d = rec["ground_truth"]["data_table"]\n    t, y = np.asarray(d["t"], float), np.asarray(d["y"], float)\n    ct = rec["chart_type"]\n    ev = []\n    # OpenML/vega dates are stored as epoch microseconds; plotted raw they give a\n    # "1.585 1e15" axis. Plot as dates. Everything below (fits, boxes) reads this\n    # same t, so evidence boxes stay in the axis\'s own coordinates.\n    is_date = t.size and np.nanmin(t) > 1e14\n    if is_date:\n        import matplotlib.dates as mdates\n        t = mdates.date2num(t.astype("datetime64[us]"))\n\n    if ct == "area":\n        ax.fill_between(t, y, np.min(y), color=colour, alpha=0.45)\n        ax.plot(t, y, color=colour, lw=1.8)\n    elif ct == "step":\n        ax.step(t, y, where="mid", color=colour, lw=1.8)\n    elif ct == "line_marker":\n        ax.plot(t, y, color=colour, lw=1.5, marker="o", ms=3.5, alpha=0.9)\n    else:\n        ax.plot(t, y, color=colour, lw=1.9)\n\n    cls = rec["label"]\n    if cls in ("ts_increasing", "ts_decreasing"):\n        b1, b0 = np.polyfit(np.arange(len(y)), y, 1)\n        fit = b1 * np.arange(len(y)) + b0\n        if not CLEAN:\n            ax.plot(t, fit, "r--", lw=1.6, alpha=0.85)\n        ev.append(("data", t[0], fit[0], t[-1], fit[-1], "fitted_trend"))\n    elif cls == "ts_changepoint":\n        ci = st.get("changepoint_index")\n        if ci is not None and 0 < ci < len(t):\n            if not CLEAN:\n                ax.axvline(t[ci], color="red", ls="--", lw=1.5, alpha=0.8)\n            span = max(1, len(t) // 25)\n            lo = t[max(0, ci - span)]\n            hi = t[min(len(t) - 1, ci + span)]\n            ev.append(("data", lo, y.min(), hi, y.max(), "changepoint"))\n            ev.append(("data", t[0], y[:ci].mean(), t[ci], y[:ci].mean(), "level_before"))\n            ev.append(("data", t[ci], y[ci:].mean(), t[-1], y[ci:].mean(), "level_after"))\n    elif cls == "ts_anomaly_spike":\n        for i in (st.get("anomaly_indices") or [])[:6]:\n            if 0 <= i < len(t):\n                if not CLEAN:\n                    ax.scatter([t[i]], [y[i]], s=110, facecolor="none", edgecolor="red", lw=2,\n                               zorder=5)\n                w = (t[-1] - t[0]) / max(len(t), 1) * 2.5\n                h = (y.max() - y.min()) * 0.06\n                ev.append(("data", t[i] - w, y[i] - h, t[i] + w, y[i] + h, "anomaly_point"))\n    elif cls == "ts_seasonal":\n        p = st.get("seasonal_period")\n        if p and p > 1:\n            step = (t[-1] - t[0]) / max(len(t) - 1, 1)\n            ev.append(("data", t[0], y.min(), t[0] + p * step, y.max(), "one_seasonal_cycle"))\n\n    if is_date:\n        loc = mdates.AutoDateLocator(maxticks=7)\n        ax.xaxis.set_major_locator(loc)\n        ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))\n    ax.set_xlabel(rec["axis_labels"]["x"])\n    ax.set_ylabel(rec["axis_labels"]["y"])\n    return ev\n\n\ndef draw_grouped(ax, rec, r, palette):\n    import seaborn as sns\n    st = rec["ground_truth"]["statistics"]\n    d = rec["ground_truth"]["data_table"]\n    g = np.asarray(d["group"])\n    v = np.asarray(d["value"], float)\n    order = sorted(set(g.tolist()))\n    ct = rec["chart_type"]\n    ev = []\n\n    if ct == "grouped_violin":\n        sns.violinplot(x=g, y=v, order=order, hue=g, hue_order=order, legend=False,\n                       palette=palette, ax=ax)\n    elif ct == "grouped_bar":\n        sns.barplot(x=g, y=v, order=order, hue=g, hue_order=order, legend=False,\n                    palette=palette, errorbar="sd", ax=ax)\n    elif ct == "pointplot":\n        sns.pointplot(x=g, y=v, order=order, color="tab:blue", errorbar="sd", ax=ax)\n    elif ct == "grouped_strip":\n        sns.stripplot(x=g, y=v, order=order, hue=g, hue_order=order, legend=False,\n                      palette=palette, alpha=0.6, ax=ax)\n    else:\n        sns.boxplot(x=g, y=v, order=order, hue=g, hue_order=order, legend=False,\n                    palette=palette, ax=ax)\n\n    meds = st.get("group_medians") or {}\n    for i, lv in enumerate(order):\n        sub = v[g == lv]\n        if len(sub):\n            ev.append(("data", i - 0.4, np.percentile(sub, 25), i + 0.4, np.percentile(sub, 75),\n                       f"group:{lv}"))\n    hi, lo = st.get("highest_group"), st.get("lowest_group")\n    for nm, tag in ((hi, "highest_median_group"), (lo, "lowest_median_group")):\n        if nm in order:\n            i = order.index(nm)\n            ev.append(("data", i - 0.45, v.min(), i + 0.45, v.max(), tag))\n\n    ax.set_xlabel(rec["axis_labels"]["x"])\n    ax.set_ylabel(rec["axis_labels"]["y"])\n    if max(len(str(o)) for o in order) > 8:\n        ax.tick_params(axis="x", rotation=20)\n    return ev\n\n\ndef draw_composition(ax, rec, r, palette):\n    st = rec["ground_truth"]["statistics"]\n    labs = np.asarray(rec["ground_truth"]["data_table"]["category"])\n    counts = st.get("counts") or {}\n    cats = list(counts)\n    vals = [counts[c] for c in cats]\n    ct = rec["chart_type"]\n    ev = []\n    cols = plt.get_cmap(palette if palette in plt.colormaps() else "tab10")(\n        np.linspace(0, 0.9, len(cats)))\n\n    if ct == "pie":\n        wedges, *_ = ax.pie(vals, labels=cats, autopct="%1.1f%%", colors=cols,\n                            startangle=int(r.integers(0, 90)))\n        ax.set_ylabel("")\n        return [("artist", wedges[0], "dominant_wedge")] + \\\n               [("artist", w, f"wedge:{c}") for w, c in list(zip(wedges, cats))[1:4]]\n    if ct == "barh":\n        bars = ax.barh(cats, vals, color=cols)\n        ax.set_xlabel("Count")\n        ax.set_ylabel(rec["axis_labels"]["x"])\n    else:\n        bars = ax.bar(cats, vals, color=cols)\n        ax.set_ylabel("Count")\n        ax.set_xlabel(rec["axis_labels"]["x"])\n        if max(len(str(c)) for c in cats) > 8:\n            ax.tick_params(axis="x", rotation=20)\n\n    out = [("artist", bars[0], "dominant_category_bar")]\n    out += [("artist", b, f"bar:{c}") for b, c in list(zip(bars, cats))[1:5]]\n    return out\n\n\nDRAW = {"univariate": draw_univariate, "bivariate": draw_bivariate,\n        "timeseries": draw_timeseries, "grouped": draw_grouped,\n        "composition": draw_composition}\n\n\ndef render_one(rec):\n    sid = rec["sample_id"]\n    r = np.random.default_rng(_seed(sid))\n    style, palette = STYLES[_seed(sid) % len(STYLES)]\n    out = IMG / rec["label"] / f"{sid}.png"\n    out.parent.mkdir(parents=True, exist_ok=True)\n    rr = np.random.default_rng(_seed(sid + "|rand")) if RANDSTYLE else None\n    rc = _rand_rc(rr) if RANDSTYLE else {}\n    if RANDSTYLE:\n        palette = str(rr.choice(RAND_CMAPS))\n\n    try:\n        with plt.style.context(style), plt.rc_context(rc):\n            fig = plt.figure(figsize=FIGSIZE, dpi=DPI)\n            if rec["chart_type"] == "jointplot":\n                # built by hand rather than seaborn.jointplot, which owns its own\n                # figure and would break the fixed canvas the boxes depend on\n                gs = GridSpec(4, 4, figure=fig, hspace=0.05, wspace=0.05)\n                ax = fig.add_subplot(gs[1:, :3])\n                axt = fig.add_subplot(gs[0, :3], sharex=ax)\n                axr = fig.add_subplot(gs[1:, 3], sharey=ax)\n                d = rec["ground_truth"]["data_table"]\n                axt.hist(np.asarray(d["x"], float), bins=28, color="tab:blue", alpha=0.7)\n                axr.hist(np.asarray(d["y"], float), bins=28, orientation="horizontal",\n                         color="tab:blue", alpha=0.7)\n                axt.axis("off")\n                axr.axis("off")\n            else:\n                ax = fig.add_subplot(111)\n\n            cmap = plt.get_cmap(palette if palette in plt.colormaps() else "tab10")\n            colour = cmap(int(r.integers(0, 8)) % cmap.N)\n            if RANDSTYLE:\n                import colorsys\n                colour = colorsys.hsv_to_rgb(rr.uniform(0, 1), rr.uniform(0.35, 1.0), rr.uniform(0.3, 0.85))\n            fn = DRAW[rec["arity"]]\n            reqs = fn(ax, rec, r, palette if rec["arity"] in ("grouped", "composition") else colour)\n\n            title = rec["axis_labels"].get("y") or rec["axis_labels"].get("x")\n            if rec["arity"] == "univariate":\n                title = f"Distribution of {rec[\'axis_labels\'][\'x\']}"\n            elif rec["arity"] == "bivariate":\n                title = f"{rec[\'axis_labels\'][\'y\']} vs {rec[\'axis_labels\'][\'x\']}"\n            elif rec["arity"] == "timeseries":\n                title = f"{rec[\'axis_labels\'][\'y\']} over {rec[\'axis_labels\'][\'x\']}"\n            elif rec["arity"] == "grouped":\n                title = f"{rec[\'axis_labels\'][\'y\']} by {rec[\'axis_labels\'][\'x\']}"\n            else:\n                title = f"Composition of {rec[\'axis_labels\'][\'x\']}"\n            if not RANDSTYLE:\n                ax.set_title(title[:78], fontsize=11, fontweight="bold")\n            elif rr.random() > 0.3:\n                ax.set_title(title[:78], fontsize=float(rr.uniform(8, 14)),\n                             fontweight=str(rr.choice(["bold", "normal"])), loc=str(rr.choice(["center", "left"])))\n\n            fig.subplots_adjust(left=0.13, right=0.965, top=0.90, bottom=0.13)\n            fig.canvas.draw()\n\n            boxes = [{"label": "plot_area",\n                      "bbox": [round(float(ax.bbox.x0), 1), round(float(H_PX - ax.bbox.y1), 1),\n                               round(float(ax.bbox.x1), 1), round(float(H_PX - ax.bbox.y0), 1)]}]\n            for req in reqs or []:\n                if req[0] == "data":\n                    _, x0, y0, x1, y1, lab = req\n                    try:\n                        boxes.append(_box_from_data(fig, ax, x0, y0, x1, y1, lab))\n                    except Exception:\n                        pass\n                else:\n                    b = _box_from_artist(fig, req[1], req[2])\n                    if b:\n                        boxes.append(b)\n\n            fig.savefig(out, dpi=DPI, facecolor="white")\n            plt.close(fig)\n    except Exception as e:\n        plt.close("all")\n        return sid, None, f"{type(e).__name__}: {e}"\n\n    return sid, {"image_size": [W_PX, H_PX], "style": style,\n                 "grounding_boxes": boxes[:14]}, None\n\n\ndef _work(chunk):\n    return [render_one(r) for r in chunk]\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--clean", action="store_true", help="no label-dependent marks")\n    ap.add_argument("--randstyle", action="store_true", help="training-only style randomisation")\n    ap.add_argument("--split", help="render only this split")\n    ap.add_argument("--arity", help="render only this arity (e.g. timeseries)")\n    ap.add_argument("--records", default=str(CACHE / "records.jsonl"),\n                    help="records to render (default cache/records.jsonl)")\n    ap.add_argument("--out", help="image dir relative to dataset/ (default images); "\n                                  "records then go to <out>/records.jsonl")\n    a = ap.parse_args()\n    global IMG, CLEAN\n    if a.out:\n        os.environ["RENDER_OUT"] = str(DATA / a.out)\n        IMG = DATA / a.out\n    if a.clean:\n        os.environ["RENDER_CLEAN"] = "1"\n        CLEAN = True\n    if a.randstyle:\n        os.environ["RENDER_RANDSTYLE"] = "1"\n\n    recs = [json.loads(l) for l in open(a.records)]\n    recs = [r for r in recs if (not a.split or r["split"] == a.split)\n            and (not a.arity or r["arity"] == a.arity)]\n    print(f"rendering {len(recs):,} charts at {W_PX}x{H_PX} "\n          f"({len(set(r[\'chart_type\'] for r in recs))} chart types) clean={CLEAN} -> {IMG}")\n    IMG.mkdir(parents=True, exist_ok=True)\n\n    from concurrent.futures import ProcessPoolExecutor, as_completed\n    chunks = [recs[i:i + 40] for i in range(0, len(recs), 40)]\n    got, errs = {}, Counter()\n    with ProcessPoolExecutor(max_workers=max(1, os.cpu_count() - 2)) as ex:\n        futs = [ex.submit(_work, c) for c in chunks]\n        bar = tqdm(as_completed(futs), total=len(futs), unit="batch",\n                   desc="rendering charts", ncols=100)\n        done = 0\n        for fu in bar:\n            for sid, meta, err in fu.result():\n                done += 1\n                if meta:\n                    got[sid] = meta\n                else:\n                    errs[err.split(":")[0]] += 1\n            bar.set_postfix(ok=len(got), failed=sum(errs.values()))\n\n    out = IMG / "records.jsonl" if a.out else DATA / "records.jsonl"\n    kept = 0\n    with open(out, "w") as fh:\n        for r in recs:\n            m = got.get(r["sample_id"])\n            if not m:\n                continue\n            r["render"] = m\n            fh.write(json.dumps(r) + "\\n")\n            kept += 1\n\n    print(f"\\nrendered {kept:,}/{len(recs):,} charts -> {IMG}")\n    if errs:\n        print("failures by type:")\n        for k, v in errs.most_common(8):\n            print(f"  {k}: {v}")\n    print(f"records with render metadata -> {out}")\n\n\nif __name__ == "__main__":\n    main()\n'

import subprocess, sys
_w = Path("/tmp/v2render"); (_w / "build").mkdir(parents=True, exist_ok=True); (_w / "cache").mkdir(exist_ok=True)
(_w / "build" / "06_render.py").write_text(RENDER_SRC)
with open(DATA / "records.jsonl") as _f, open(_w / "cache" / "records.jsonl", "w") as _g:
    for _l in _f:
        if '"arity": "timeseries"' in _l: _g.write(_l)
OV = _w / "dataset" / "images_v2ts"
if not OV.exists():
    subprocess.run([sys.executable, str(_w / "build" / "06_render.py"), "--arity", "timeseries",
                    "--clean", "--out", "images_v2ts"], check=True)

from torch.utils.data import WeightedRandomSampler
for _df in (tr, va, te):
    _m = _df.arity == "timeseries"
    _df.loc[_m, "image"] = [str(OV / p.split("/", 1)[1]) for p in _df.loc[_m, "image"]]
_ts = pd.concat([tr, va, te]).query("arity == 'timeseries'").image
assert len(_ts) == 5000 and all(Path(p).exists() for p in _ts), "overlay incomplete"
print(f"v2 overlay: {len(_ts):,} clean time-series images from {OV}")
# The metadata frame carries every chart's full data table (up to 58k points each).
# DataLoader workers fork the process and slowly copy it; NB5 v1 and v3 both died of
# host OOM early in epoch 2. Training needs four columns.
_KEEP = ["sample_id", "image", "label", "provenance"]
tr, va, te = (d[_KEEP].copy() for d in (tr, va, te))
del meta, _ts; gc.collect()
import psutil

# + supplementary real pool: verified real candidates the balanced set left out,
# from TRAIN-split sources only (build/12_extra_real.py asserts no val/test source)
_ex = [p for p in Path("/kaggle/input").glob("**/extra_real.csv")]
assert _ex, "attach dataset asifxzaman/insightvlm-extra-real"
_x = pd.read_csv(_ex[0])
# Kaggle nests an uploaded zip one level deeper (images_extra/images_extra/...), so
# locate the folder holding the class directories from one known file
_hit = next(Path("/kaggle/input").glob(f"**/{_x.label.iloc[0]}/{_x.sample_id.iloc[0]}.png"))
_x["image"] = [str(_hit.parent.parent / l / f"{s}.png") for l, s in zip(_x.label, _x.sample_id)]
assert all(Path(p).exists() for p in _x.image), "extra images missing"
tr = pd.concat([tr, _x[_KEEP]], ignore_index=True)
print(f"+ {len(_x):,} extra real training charts -> train {len(tr):,}")

# + renderer augmentation: every TRAINING chart (train + extra) also exists as a Plotly
# render and a randomised-style matplotlib render. Views share sample ids, so each is
# located under its own folder name.
_ac = [p for p in Path("/kaggle/input").glob("**/aug_train.csv")]
assert _ac, "attach dataset asifxzaman/insightvlm-aug-train"
_a = pd.read_csv(_ac[0])
VIEWS = ["plotly", "mplrand"]
for _v in VIEWS:
    _g = _a[_a.view == _v]
    _hit = next(Path("/kaggle/input").glob(f"**/images_{_v}_train/{_g.label.iloc[0]}/{_g.sample_id.iloc[0]}.png"))
    _m = {s: str(_hit.parent.parent / l / f"{s}.png") for l, s in zip(_g.label, _g.sample_id)}
    tr["img_" + _v] = tr.sample_id.map(_m)
    assert tr["img_" + _v].notna().mean() > 0.99, f"{_v}: only {tr['img_' + _v].notna().mean():.1%} covered"
    assert Path(tr["img_" + _v].dropna().iloc[-1]).exists()
assert not set(_a.sample_id) & (set(te.sample_id) | set(va.sample_id)), "augmentation touches val/test"

class RendererAugDS(ChartDS):
    # original matplotlib render or one of the alternative views, uniformly
    def __getitem__(self, i):
        r = self.df.iloc[i]
        opts = [DATA / r.image] + [r["img_" + v] for v in VIEWS if isinstance(r["img_" + v], str)]
        img = Image.open(opts[random.randrange(len(opts))]).convert("RGB")
        return self.tf(img), C2I[r.label], i
TRAIN_DS = RendererAugDS
print(f"renderer augmentation: {len(tr):,} train charts x (matplotlib + {' + '.join(VIEWS)})")

TRAIN_DS = globals().get("TRAIN_DS", ChartDS)
is_real = (tr.provenance == "real").values
va_real = (va.provenance == "real").values
te_real = (te.provenance == "real").values
print(f"train real {is_real.sum():,} synth {(~is_real).sum():,}")

@torch.no_grad()
def probs(model, df, size, bs):
    model.eval(); out = np.zeros((len(df), NC), np.float32)
    for x, _, ix in loader(df, size, False, bs):
        with torch.autocast("cuda", enabled=DEVICE.type=="cuda"):
            o = model(x.to(DEVICE, non_blocking=True))
        out[ix.numpy()] = torch.softmax(o.float(), 1).cpu().numpy()
    return out

def train_real(name, tag, size=384, epochs=12, bs=24, lr=3e-4, real_share=0.5, patience=4):
    # real_share=None: plain shuffle (natural ~31% real), the sampling ablation
    ds = TRAIN_DS(tr, size, train=True)
    if real_share is None:
        samp = dict(shuffle=True)
    else:
        w = np.where(is_real, real_share / is_real.sum(), (1 - real_share) / (~is_real).sum())
        samp = dict(sampler=WeightedRandomSampler(torch.tensor(w), len(tr), replacement=True))
    # 2 workers, as in NB1-3: 4 workers + ConvNeXt killed the NB5 v1 kernel
    dl_tr = DataLoader(ds, batch_size=bs, num_workers=2, prefetch_factor=2, **samp)
    model = build(name).to(DEVICE)
    if torch.cuda.device_count() > 1: model = nn.DataParallel(model)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=epochs,
                                                steps_per_epoch=len(dl_tr), pct_start=0.15)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type=="cuda")
    ckpt = f"{tag}_best.pt"; best = (-1, -1); stale = 0; hist = []
    for ep in range(1, epochs+1):
        model.train(); t0 = time.time(); tot=n=corr=0
        for bi, (x, y, _) in enumerate(dl_tr):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=DEVICE.type=="cuda"):
                out = model(x); loss = crit(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item()*y.size(0); corr += (out.argmax(1)==y).sum().item(); n += y.size(0)
            if bi % 60 == 0:
                print(f"  ep{ep} [{bi}/{len(dl_tr)}] loss {tot/n:.3f} acc {corr/n:.3f}", flush=True)
        pv = probs(model, va, size, bs).argmax(1); ok = pv == va.label.map(C2I).values
        vr, vm = float(ok[va_real].mean()), float(ok.mean())
        hist.append(dict(epoch=ep, loss=tot/n, train_acc=corr/n, val_real=vr, val_acc=vm))
        print(f"  ep{ep} loss {tot/n:.3f} train {corr/n:.3f} val_real {vr:.4f} val_acc {vm:.4f} "
              f"({time.time()-t0:.0f}s, RAM {psutil.virtual_memory().percent:.0f}%)", flush=True)
        if (vr, vm) > best:   # real-only first, mixed breaks ties
            best, stale = (vr, vm), 0
            torch.save((model.module if isinstance(model, nn.DataParallel) else model).state_dict(), ckpt)
        else:
            stale += 1
            if stale >= patience: print("  early stop"); break
    del model, opt, dl_tr; gc.collect(); torch.cuda.empty_cache()
    m = build(name).to(DEVICE); m.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    if torch.cuda.device_count() > 1: m = nn.DataParallel(m)
    pv_, pt_ = probs(m, va, size, bs), probs(m, te, size, bs)
    np.savez(f"{tag}_probs.npz", val=pv_, test=pt_)
    okt = pt_.argmax(1) == te.label.map(C2I).values
    r = dict(model=name, tag=tag, size=size, real_share=real_share, best_val_real=best[0], best_val_acc=best[1],
             test_acc=float(okt.mean()), test_real=float(okt[te_real].mean()),
             test_synth=float(okt[~te_real].mean()), history=hist)
    print(f"== {tag}: TEST acc {r['test_acc']:.4f}  real {r['test_real']:.4f}  synth {r['test_synth']:.4f}")
    del m; gc.collect(); torch.cuda.empty_cache()
    return r

In [ ]:
RUNS = [('densenet121', 'd121_384_real50_xp', 0.5)]
RESULTS = {}
for name, tag, share in RUNS:
    t0 = time.time()
    RESULTS[tag] = train_real(name, tag, real_share=share)
    RESULTS[tag]["minutes"] = round((time.time()-t0)/60, 1)
    save(RESULTS, "nb5_results.json")   # after each run, so a timeout keeps finished ones